# 서울시 부동산 2024 — 전처리(판다스) 4회차 문제


이번 회차는 **머신러닝 입력(X) 만들기** 연습입니다.  
서울시 부동산 데이터로 `신고구분(직거래/중개거래)`를 예측하는 걸 목표로 전처리를 쭉 이어서 해봅시다.

오늘 포인트
- Split -> Fit -> Transform (누수 방지)
- 결측 플래그 만들기
- 그룹 중앙값으로 결측 채우기(층)
- 이상치(IQR 캡핑) + log1p
- 파생변수(단가/연식 등)
- get_dummies + reindex로 컬럼 정렬
- StandardScaler train만 fit


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv("2024.csv", encoding="cp949", low_memory=False)

# 실습 속도 조절용(원하면 사용)
# df = df.sample(50_000, random_state=42).reset_index(drop=True)

print(df.shape)
df.head()


---

### 문제 1 — 타깃(y) 만들기 + SPLIT
이번에는 `신고구분`으로 **직거래(1) / 중개거래(0)** 이진 분류를 해봅시다.

1) 먼저 `신고구분`이 결측인 행은 제거하세요. (`dropna`)
2) 타깃 만들기
- `y = (df['신고구분'] == '직거래').astype(int)`
3) 피처 만들기
- `X = df.drop(columns=['신고구분'])`
4) `train_test_split`
- `test_size=0.2`, `random_state=42`, `stratify=y`


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 2 — 날짜 처리: `계약일` -> datetime + 파생컬럼
`계약일`이 20241014 같은 형태로 들어있습니다.

- `계약일_dt` 컬럼을 만드세요.
  - `pd.to_datetime(..., format='%Y%m%d', errors='coerce')`
- 아래 파생컬럼도 만들어봅시다.
  - `계약월` (`.dt.month`)
  - `계약일자` (`.dt.day`)
  - `계약요일` (`.dt.dayofweek`, 월=0)

> train/test 둘 다에 똑같이 적용 (여긴 fit 단계가 없습니다)


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 3 — 결측 플래그 만들기
결측 자체가 의미가 있을 수 있으니, 아래 컬럼들에 대해 결측 플래그를 만들어봅시다.

- 대상 컬럼
  - `토지면적(㎡)`
  - `층`
  - `건축년도`
  - `건물명`
  - `신고한 개업공인중개사 시군구명`

각 컬럼에 대해 `컬럼명_isna`를 만들고 0/1로 채우세요.


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 4 — 그룹 중앙값으로 `층` 결측 채우기 (누수 없는 버전)
`층`은 결측이 꽤 있습니다. Titanic의 Age처럼 **그룹 중앙값**으로 채워봅시다.

- 그룹 기준: `['자치구명', '건물용도']`
- `층_med_table` (중앙값 테이블)은 **train에서만** 만들기
- train/test에 `merge`로 붙인 뒤 결측을 채우고, 임시 컬럼은 제거


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 5 — 나머지 결측 채우기 (train 기준값으로!)
아래처럼 **train에서 기준값을 만든 뒤**, train/test에 동일 적용하세요.

- `토지면적(㎡)` : train 중앙값
- `건축년도` : train 중앙값
- `지번구분명` : train 최빈값
- `자치구명` : 결측이면 `'미상'`
- `건물명` : 결측이면 `'미상'`
- `신고한 개업공인중개사 시군구명` : 결측이면 `'미상'`

> 숫자형 중앙값은 `median()`, 범주형 최빈값은 `mode()[0]`


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 6 — 이상치 캡핑 + log1p
부동산 데이터는 금액/면적에 극단값이 있습니다.

- `물건금액(만원)`을 train 기준으로 IQR 캡핑해서 `가격_cap` 만들기
- `건물면적(㎡)`도 train 기준으로 IQR 캡핑해서 `면적_cap` 만들기
- 각각 `log1p` 변환 컬럼도 만들기
  - `가격_log1p = np.log1p(가격_cap)`
  - `면적_log1p = np.log1p(면적_cap)`

> lo/hi 경계는 train에서만 계산!


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
# q1, q3 = X_train["물건금액(만원)"].quantile([0.25, 0.75])
# iqr = q3 - q1
# lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr


---

### 문제 7 — 파생변수 만들기 (가격/면적/연식)
모델이 좋아할 만한 파생변수를 몇 개 추가해봅시다.

1) `price_per_sqm`
- `가격_cap / 면적_cap`

2) `건축연령`
- `계약일_dt`의 연도 - `건축년도`
  - (`계약일_dt`가 NaT인 경우를 대비해서 연도 결측은 2024로 채워도 OK)

3) `is_high_floor`
- `층 >= 10` 이면 1, 아니면 0

4) `land_ratio`
- `토지면적(㎡) / 면적_cap`


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 8 — 희귀 범주 묶기 + 원-핫 인코딩 + 컬럼 정렬
범주가 너무 많으면 원-핫이 폭발합니다. **상위 몇 개만 남기고 나머지는 기타로** 묶어봅시다.

1) `법정동명_top`
- train에서 `법정동명` 상위 20개만 남기고 나머지는 `'기타'`
- 같은 규칙을 test에도 적용

2) 원-핫 인코딩
- 대상 컬럼(예시): `자치구명`, `건물용도`, `지번구분명`, `법정동명_top`
- `get_dummies(drop_first=True)`
- test는 **train 컬럼 기준**으로 `reindex(fill_value=0)`

3) (중요) 누수 컬럼 제거
- `신고한 개업공인중개사 시군구명`과 그 결측 플래그는 **정답 힌트에 가깝기 때문에** 마지막 모델 입력에서는 제외하세요.

추가 힌트
- `건물명`은 고유값이 너무 많아서(거의 ID 수준) 보통은 **drop**하는 게 안전합니다.


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
# 모델에서 빼도 되는 원본 컬럼 정리
# drop_cols = [
#     "접수연도", "자치구코드", "법정동코드",
#     "계약일", "계약일_dt", "법정동명",
#     "신고한 개업공인중개사 시군구명",
#     "신고한 개업공인중개사 시군구명_isna",  # 누수 가능성이 매우 높음
#     "지번구분", "본번", "부번",
#     "취소일", "권리구분",
#     "건물명",  # 고유값 너무 많음
#     "물건금액(만원)", "건물면적(㎡)",  # log/cap 버전으로 대체
#     "가격_cap", "면적_cap"  # 파생변수 만들었으면 원본은 굳이
# ]

---

### 문제 9 — 스케일링
`StandardScaler`로 수치형만 스케일링
- train: fit_transform
- test: transform


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
